<div style="width: 100%; clear: both;">
<div style="float: left; width: 50%;">
<img src="https://www.uoc.edu/content/dam/news/images/noticies/2016/202-nova-marca-uoc.jpg" align="left" width="45%">
</div>
<div style="float: right; width: 50%;">
<p style="margin: 0; padding-top: 22px; text-align:right;">M2.878 · Trabajo de Fin de Máster · Evaluación del impacto económico</p>
<p style="margin: 0; text-align:right;">2025-2 · Máster universitario en Ciencia de datos</p>
<p style="margin: 0; text-align:right; padding-button: 100px;">Marcos Rodríguez Soler</p>
</div>
</div>
<div style="width:100%;">&nbsp;</div>

# Evaluación del impacto económico

En este _notebook_ se lleva a cabo la evaluación del impacto económico de los modelos de aprendizaje automático entrenados, donde primero se seleccionan los mejores modelos de aprendizaje automático entrenados, y posteriormente se realiza una comparación con el rendimiento que presenta el modelo _naive_ que predice la demanda futura como el número de ventas en el último periodo registrado. En ambos casos, los modelos se ponen a prueba en una política de reposición _Order-Up-To (R, S)_.

<ol style="list-style: none; padding-left: 0;">
    <li>1. <a href="#ej1">Rendimiento de los modelos</a></li>
    <li>2. <a href="#ej2">Simulación <i>Order-Up-To (R, S)</i></a></li>
    &nbsp;&nbsp;2.1. <a href="#ej2.1">Mejor modelo global</a> <br>
    &nbsp;&nbsp;2.2. <a href="#ej2.2">Mejores modelos por <i>cluster</i></a> <br>
    &nbsp;&nbsp;2.3. <a href="#ej2.3">Modelo <i>naive</i></a> <br>
    <li>3. <a href="#ej3">Tabla resumen</a></li>
</ol>

In [ ]:
import math
import time
import random

import pandas as pd
import numpy as np

from scipy.stats import norm

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

from typing import Dict, List, Tuple

pd.set_option("display.max_columns", None)
%matplotlib inline

<br><br><a id="ej1"></a>
# 1. Rendimiento de los modelos

A lo largo de todo el proyecto se han entrenado un total de 15 modelos de aprendizaje automático, los cuales se listan a continuación.

<ol style="list-style: none; padding-left: 0;">
    <li>Modelos <i>Random Forest</i></li>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;├── Modelo global <br>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;├── Modelo del <i>cluster top_ventas</i> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;├── Modelo del <i>cluster residual</i> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;├── Modelo del <i>cluster alta_rotacion</i> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;└── Modelo del <i>cluster estandar</i> <br>
    <li>Modelos <i>XGBoost</i></li>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;├── Modelo global <br>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;├── Modelo del <i>cluster top_ventas</i> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;├── Modelo del <i>cluster residual</i> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;├── Modelo del <i>cluster alta_rotacion</i> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;└── Modelo del <i>cluster estandar</i> <br>
    <li>Modelos <i>LSTM</i></li>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;├── Modelo global <br>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;├── Modelo del <i>cluster top_ventas</i> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;├── Modelo del <i>cluster residual</i> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;├── Modelo del <i>cluster alta_rotacion</i> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;└── Modelo del <i>cluster estandar</i> <br>
</ol>

Para la evaluación y comparación de los modelos se computa el RMSE de cada uno de ellos. Se considera que el mejor modelo es aquel cuyo RMSE global sea el más bajo de todos. El indicador se calcula para cada combinación de producto, $p$, y horizonte temporal, $h$. Posteriormente, para cada producto se promedian únicamente los RMSE correspondientes a los horizontes comprendidos dentro del ciclo de aprovisionamiento de cada producto, definido como la suma del tiempo de entrega, $L$, y los días entre pedidos, $R$. Esto se debe a que la demanda relevante de cada artículo es la que corresponde a todo el intervalo de tiempo $R + L$, de forma que los errores cometidos en los horizontes posteriores no deben tenerse en cuenta.

$RMSE_p = \frac{1}{R+L} \sum_{h=1}^{R+L} RMSE_{p,h}$

Finalmente, el RMSE global de cada modelo se obtiene promediando los errores operativos de todos los productos, $N$.

$RMSE_{modelo} = \frac{1}{N} \sum_{p=1}^{N} RMSE_p$

Para encontrar este mejor algoritmo, primero se cargan los resultados de cada modelo entrenado y los archivos _csv_ con los RMSEs a nivel de horizonte y producto de cada modelo empleando la función _**importar_resultados()**_. Para la correcta ejecución de esta función, los archivos con los resultados y los errores se deben encontrar en una carpeta de nombre _MODELOS_ situada en la misma jerarquía que el directorio que contiene este _notebook_. 

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;├── MODELOS <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;└── DIRECTORIO_ACTUAL

Los nombres de los archivos con los resultados siguen la estructura _res_modelo_grupo.csv_, donde _modelo_ puede ser _random_forest_, _xgboost_, _lstm_ o _naive_, y _grupo_ puede ser _global_, _top_ventas_, _residual_, _alta_rotacion_ o _estandar_. De la misma forma, los ficheros con los RMSEs a nivel de horizonte por producto siguen la estructura _rmses_modelo_grupo.csv_.

Seguidamente, se utiliza la función _**rendimiento_modelo()**_ para obtener el promedio de los RMSEs relevantes de cada producto para cada modelo, con el objetivo de estimar el error de cada algoritmo entrenado.

In [ ]:
# Función para importar los resultados de los modelos
def importar_resultados(modelo: str, grupo: str) -> Tuple[pd.DataFrame]:
    """Se importan los resultados y los RMSE por horizonte y producto de un modelo concreto para un grupo determinado

    Argumentos:
        modelo (str) -> Nombre del modelo del cual se desea recuperar los resultados en minúsculas y con _ en los espacios
        grupo (str) -> Nombre del grupo al cual se ha aplicado el modelo del cual se desea recuperar los resultados en
        minúsculas y con _ en los espacios

    Devuelve:
        Tuple[pd.DataFrame] -> Tupla con los DataFrames de los resultados y los RMSE por producto y horizonte temporal
    """
    ruta_resultados: str = f"../MODELOS/res_{modelo}_{grupo}.csv"
    ruta_rmses: str = f"../MODELOS/rmses_{modelo}_{grupo}.csv"
    
    resultados: pd.DataFrame = pd.read_csv(ruta_resultados).drop("Unnamed: 0", axis=1)
    rmses: pd.DataFrame = pd.read_csv(ruta_rmses).drop("Unnamed: 0", axis=1)

    return resultados, rmses

In [ ]:
# Función para estimar el rendimiento de cada modelo
def rendimiento_modelo(resultados_modelo: pd.DataFrame, rmses_modelo: pd.DataFrame) -> float:
    """Devuelve el RMSE promedio de un modelo de aprendizaje automático. Para el cálculo se escoge el RMSE
    correspondiente con el ciclo de aprovisionamiento de cada producto.

    Argumentos:
        resultados_modelo (pd.DataFrame) -> DataFrame con los resultados del conjunto de prueba de un modelo concreto
        rmses_modelo (pd.DataFrame) -> DataFrame con los RMSEs de todas las combinaciones de productos y horizontes temporales

    Devuelve:
        float -> RMSE promedio del modelo
    """
    rmses_relevantes: List[float] = []

    for producto, df_producto in resultados_modelo.groupby("producto"):

        ciclo_aprov: int = int(  # Ciclo aprovisionamiento
            df_producto["diasLeadtime"].mean() +
            df_producto["diasEntrePedidos"].mean()
        )

        fila_rmse: pd.Series = rmses_modelo.loc[  # RMSEs del producto
            rmses_modelo["producto"] == producto
        ].iloc[0]

        columnas_horizontes: List[str] = [  # Horizontes temporales relevantes
            f"hor_{h}" for h in range(1, ciclo_aprov + 1)
        ]

        rmse_producto: float = fila_rmse[columnas_horizontes].mean()  # Promedio RMSEs relevantes

        rmses_relevantes.append(rmse_producto)

    return np.mean(rmses_relevantes)

In [ ]:
# Se calculan los resultados de todos los modelos y se calcula el rendimiento de cada uno de ellos
nombres_modelos: List[str] = ["random_forest", "xgboost", "lstm", "naive"]
nombres_grupos: List[str] = ["global", "top_ventas", "residual", "alta_rotacion", "estandar"]

rendimientos: Dict[str, float] =  {}
rmses_modelos: Dict[str, pd.DataFrame] = {}
modelos: Dict[str, pd.DataFrame] = {}

for nombre_modelo in nombres_modelos:
    for nombre_grupo in nombres_grupos:
        
        if nombre_modelo == "naive" and nombre_grupo != "global":
            continue

        resultados, rmses = importar_resultados(nombre_modelo, nombre_grupo)
        modelos[f"{nombre_modelo}_{nombre_grupo}"] = resultados
        rmses_modelos[f"{nombre_modelo}_{nombre_grupo}"] = rmses
        
        rendimiento: float = rendimiento_modelo(resultados, rmses)
        rendimientos[f"{nombre_modelo}_{nombre_grupo}"] = rendimiento

In [ ]:
# Se representan los promedios de los RMSEs de cada modelo
plt.figure(figsize=(8, 6))
plt.plot(rendimientos.keys(), rendimientos.values(), color="blue")
plt.scatter(rendimientos.keys(), rendimientos.values(), color="blue")

plt.title("Promedio de los RMSEs de cada modelo")
plt.xlabel("Modelos")
plt.xticks(rotation=45, ha="right")
plt.ylabel("RMSE promedio")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

A modo de resumen, el gráfico _Promedio de los RMSEs de cada modelo_ muestra el RMSE promedio de cada modelo calculado a partir del horizonte temporal relevante de cada producto, es decir, el correspondiente a su ciclo de reposición. En términos generales, se observa que las tendencias de error son muy similares independientemente del algoritmo empleado, ya que los modelos _Random Forest_, _XGBoost_ y LSTM presentan comportamientos prácticamente equivalentes en cada uno de los grupos de productos. Este resultado sugiere que la dificultad del problema depende en gran medida de las características intrínsecas de cada _cluster_, más que del modelo concreto utilizado.

En particular, el grupo _estandar_ presenta los errores más bajos, lo que indica que estos productos muestran patrones de demanda relativamente estables, homogéneos, y sencillos de predecir, lo que se alinea con la naturaleza de este _cluster_. Del mismo modo, el _cluster_ _alta_rotacion_ también obtiene resultados favorables, aunque con una mayor variabilidad en la demanda. Por otro lado, el grupo _top_ventas_ concentra los errores más elevados en todos los modelos. Este comportamiento puede explicarse por la presencia de picos de demanda muy pronunciados y difíciles de anticipar utilizando únicamente las variables disponibles en el conjunto de datos. En este sentido, gran parte de las variables derivadas utilizadas durante el entrenamiento se basan en estadísticas de ventana móvil, como medias o máximos recientes, las cuales tienden a suavizar las predicciones y dificultan la detección de cambios bruscos en la demanda. Como conclusión, se observa que los modelos entrenados específicamente para cada _cluster_ tienden a obtener errores inferiores a los modelos globales, hecho que refuerza la hipótesis planteada sobre la existencia de patrones de comportamiento diferenciados entre grupos de productos.

Finalmente, el modelo _naive_ presenta un error superior a la mayoría de modelos de aprendizaje automático, lo que evidencia que los algoritmos empleados son capaces de capturar parte de la dinámica temporal de la demanda más allá de simplemente extrapolar el último valor observado.

Por otro lado, en base a los resultados comentados, como resulta complicado establecer si la agrupación en _clusters_ supone una mejora respecto a un enfoque global, se considera oportuno seleccionar tanto el mejor modelo global como los mejores modelos de cada _cluster_ para la evaluación de costes. El objetivo es averiguar en términos de impacto económico si la segmentación de productos mejora el rendimiento operativo frente a un único modelo global, y a su vez comparar los rendimientos con los del modelo _naive_.

In [ ]:
# Se define la función mejor_modelo() para encontrar el modelo con el mejor rendimiento de cada tipo
def mejor_modelo(rendimientos: Dict[str, float], grupo: str) -> Tuple[str, float]:
    """Devuelve el modelo con el mejor rendimiento de todos los que se han entrenado para el grupo especificado

    Argumentos:
        rendimientos (Dict[str, float]) -> Diccionario con los RMSEs de cada modelo entrenado
        grupo (str) -> Nombre del grupo del que se desea encontrar el mejor modelo

    Devuelve:
        Tuple[str, float] -> Tupla con el nombre del mejor modelo del grupo especificado y su RMSE
    """
    mejor_modelo: Tuple[str, float] = ("", np.inf)
    for nombre, error in rendimientos.items():
        if grupo not in nombre:
            continue
    
        if error < mejor_modelo[1]:
            mejor_modelo = (nombre, error)

    return mejor_modelo

In [ ]:
# Selección del mejor modelo global
res_mejor_modelo_global: Tuple[str, float] = mejor_modelo(rendimientos, "global")
mejor_modelo_global: pd.DataFrame = modelos[res_mejor_modelo_global[0]]
rmses_mejor_modelo_global: pd.DataFrame = rmses_modelos[res_mejor_modelo_global[0]]

print(f"El modelo global con el RMSE más bajo es {res_mejor_modelo_global[0]} con RMSE={res_mejor_modelo_global[1]:.2f} unidades")

In [ ]:
# Selección de los mejores modelos por cluster
res_mejores_modelos_grupos: Dict[str, Tuple[str, float]] = {}
for nombre_grupo in nombres_grupos:
    if nombre_grupo == "global":
        continue

    res_mejores_modelos_grupos[nombre_grupo] = mejor_modelo(rendimientos, nombre_grupo)

for grupo, res_mejor_modelo_grupo in res_mejores_modelos_grupos.items():
    print(f"El modelo del cluster {grupo} con el RMSE más bajo es {res_mejor_modelo_grupo[0]} con RMSE={res_mejor_modelo_grupo[1]:.2f} unidades")

In [ ]:
# Fusión de los mejores resultados de cada cluster en un solo DataFrame
lista_res_modelos_grupos: List[pd.DataFrame] = [modelos[modelo[0]] for modelo in res_mejores_modelos_grupos.values()]
lista_rmses_modelos_grupos: List[pd.DataFrame] = [rmses_modelos[modelo[0]] for modelo in res_mejores_modelos_grupos.values()]

mejores_modelos_grupos: pd.DataFrame = pd.concat(lista_res_modelos_grupos, ignore_index=True)
rmses_mejores_modelos_grupos: pd.DataFrame = pd.concat(lista_rmses_modelos_grupos, ignore_index=True)

De 3 de los 4 _clusters_ totales en los que se han segmentado los datos, el modelo que ha obtenido el mejor rendimiento es _Random Forest_, que tanto metodológicamente como conceptualmente resulta más sencillo y simple que los modelos _XGBoost_ y sobretodo LSTM. Este hecho demuestra que la implementación más simple muchas veces suele ser la mejor de todas, lo cual resulta especialmente relevante si se desea escalar el proyecto o desplegar los modelos en un entorno operativo.

<br><br><a id="ej2"></a>
# 2. Simulación _Order-Up-To (R, S)_
Una vez evaluado el rendimiento predictivo de los distintos modelos de aprendizaje automático, el siguiente paso consiste en analizar cómo los resultados obtenidos afectan al rendimiento económico en un sistema de gestión de inventario real. En este sentido, no basta únicamente con comparar métricas de evaluación del error, ya que un mismo nivel de error puede traducirse en impactos económicos muy distintos dependiendo de cómo se gestionen las reposiciones y las roturas de _stock_. Por este motivo, se implementa una simulación siguiendo una política de inventario de tipo _Order-Up-To (R, S)_. En este enfoque, el inventario no se monitoriza de forma continua, sino únicamente cada $R$ periodos de tiempo. En cada revisión, se calcula la posición de inventario actual y se realiza un pedido cuyo objetivo es elevar el nivel total de inventario hasta un valor prefijado $S$, denominado nivel objetivo. Dicho nivel debe ser suficiente para cubrir tanto la demanda esperada durante el ciclo de aprovisionamiento como la incertidumbre asociada a dicha demanda.

En la simulación implementada, el intervalo de revisión $R$ corresponde a la variable _diasEntrePedidos_, mientras que el tiempo de entrega del proveedor viene determinado por _diasLeadtime_, $L$, donde la suma de ambos valores equivale al ciclo de aprovisionamiento completo del producto.

$ciclo\_aprov = diasLeadtime + diasEntrePedidos$

Este valor representa el horizonte temporal durante el cual el inventario debe ser capaz de satisfacer la demanda antes de recibir el siguiente pedido. Asimismo, el nivel objetivo $S$ se calcula como la suma de la demanda prevista durante el ciclo de aprovisionamiento, $\hat{y}_{R+L}$, y un _stock_ de seguridad, $SS$, para absorber la incertidumbre de las predicciones.

$S = \hat{y}_{R+L} + SS$

En cuanto al inventario de seguridad, este se calcula utilizando el RMSE asociado al horizonte temporal correspondiente al ciclo de aprovisionamiento del producto, es decir, $R + L$. En particular, dicho RMSE se emplea como una aproximación de la desviación estándar del error de predicción, permitiendo estimar la incertidumbre asociada a la demanda futura. De este modo, el _stock_ de seguridad se obtiene mediante la expresión:

$SS = k \cdot \sigma_{R+L} \cdot \sqrt{R+L}$

Donde $k$ es el factor asociado al nivel de servicio deseado, $\sigma_{R+L}$ representa la desviación estándar aproximada de los errores de predicción, y $R + L$ corresponde al ciclo de aprovisionamiento completo.

Para llevar a cabo la simulación, se ha definido la función _**simulacion_order_up_to()**_, donde se recorre secuencialmente cada día del conjunto de prueba de un producto concreto. En cada iteración se comprueba inicialmente si existe algún pedido pendiente de recepción cuyo plazo de entrega haya finalizado, actualizando el inventario disponible en consecuencia. Posteriormente, se descuenta la demanda real observada ese día. Si la demanda supera el _stock_ disponible, se produce una rotura de inventario, registrándose tanto las ventas realizadas como las ventas perdidas asociadas.

Además, en aquellos días donde corresponde realizar una revisión del inventario, se calcula la posición de inventario actual, es decir, la cantidad de inventario de la que se dispone. Esta magnitud se calcula como la suma del _stock_ disponible y las unidades pendientes de recibir. A continuación, se estima la demanda futura utilizando las predicciones generadas previamente por el modelo de aprendizaje automático para todos los horizontes comprendidos dentro del ciclo de aprovisionamiento. Finalmente, se calcula la cantidad óptima a pedir como la diferencia entre el nivel objetivo y la posición de inventario actual, donde el nivel objetivo es la suma de la demanda esperada y el _stock_ de seguridad calculado.

Durante toda la simulación también se contabilizan dos tipos de costes operativos. Por un lado, se calcula diariamente un coste de mantenimiento proporcional al valor del inventario almacenado. Este coste debe tener en cuenta el propio coste de almacenaje, asociado al alquiler y el mantenimiento de las naves del almacén, y el coste de oportunidad de inversión, asociado al dinero que se deja de ganar al tenerlo invertido en inventario. Este coste se puede aproximar con la siguiente fórmula, que es el 5% del producto del precio medio del artículo, que equivale a la variable _eurPrecioMedio_ del conjunto de datos, $P$, y el número de unidades de dicho ítem almacenadas, $U$.

$coste\_diario = 0.05 \cdot P \cdot U$

Por otro lado, las roturas de _stock_ generan un coste asociado a las ventas perdidas, donde en este caso se ha aproximado al precio medio del producto, $P$, multiplicado por el número de ventas perdidas, $y_{perdidas}$. 

$coste\_rotura\_stock = P \cdot y_{perdidas}$

Adicionalmente, también se registran métricas agregadas como las ventas totales realizadas y la demanda total observada, lo que permite analizar posteriormente el nivel de servicio alcanzado por cada estrategia de predicción.

Asimismo, la simulación requiere definir un nivel inicial de inventario para cada producto. En este caso, el _stock_ inicial se aproxima mediante la suma de las ventas reales observadas durante el primer ciclo de aprovisionamiento del conjunto de prueba. Esta decisión permite inicializar el sistema con un nivel de inventario coherente con la demanda esperada del producto, evitando introducir sesgos artificiales derivados de comenzar la simulación con un _stock_ nulo o arbitrario. De este modo, la evolución posterior del inventario depende principalmente de la calidad de las predicciones y de la política de reposición aplicada.

In [ ]:
def simulacion_order_up_to(df_producto: pd.DataFrame, df_rmses_producto: pd.DataFrame, nivel_servicio: float) -> Tuple[List[float], Dict[str, float]]:
    """Se calculan las unidades de un producto cada día a lo largo del periodo del conjunto de prueba del mismo producto.
    La función simula una política de reposición de tipo order-up-to (R, S), por lo que tiene en cuenta los tiempos de entrega,
    los días entre pedidos, el nivel objetivo y la posición de inventario de cada ciclo de reposición. Adicionalmente, también
    sirve para calcular tanto los costes asociados a la presencia de stock en el almacén como los costes de las roturas de stock.
    La función también devuelve cuatro métricas útiles para entender el rendimiento del modelo utilizado para las prediccines de la
    demanda. Entre ellas se incluyen el coste total del almacenamiento del inventario del producto para toda la simulación, el coste
    asociado a las rupturas de stock durante la simulación, las ventas totales que se han efectuado, y la demanda total que sse ha
    observado.

    Argumentos:
        df_producto (pd.DataFrame) -> DataFrame con las predicciones de la demanda de un producto concreto para su conjunto de prueba
        df_rmses_producto (pd.DataFrame) -> DataFrame con los RMSE de cada horizonte temporal de un producto concreto
        nivel_servicio (float) -> Porcentaje de la demanda que se desea satisfacer

    Devuelve:
        List[float] -> Número de unidades del producto de las que se dispone al final de un día
        Dict[str , float] -> Diccionario con las cuatro métricas de evaluación de la simulación
    """
    # Inicialización de las métricas de evaluación de la simulación económica
    coste_stock_total: float = 0.0
    coste_rotura_total: float = 0.0
    ventas_totales: float = 0.0
    demanda_total: float = 0.0

    # Inicialización de parámetros de la simulación
    factor_servicio: float = norm.ppf(nivel_servicio)
    leadtime: int = int(df_producto["diasLeadtime"].mean())
    dias_entre_pedidos: int = int(df_producto["diasEntrePedidos"].mean())
    ciclo_aprov: int = leadtime + dias_entre_pedidos
    precio: float = df_producto["eurPrecioMedio"].mean()
    columnas_horizontes: List[str] = [f"hor_{h}" for h in range(1, ciclo_aprov + 1)]
    sigma: float = df_rmses_producto[columnas_horizontes].values[0].mean()

    # Inicialización del inventario inicial, de la lista con el stock de cada día de la simulación, y de la
    # lista con los pedidos pendientes de ser recibidos
    stock_actual: float = df_producto["udsVenta"].iloc[:ciclo_aprov].sum()
    unidades: List[float] = []
    pedidos_pendientes: List[tuple] = []

    # Se recorre cada día de la simulación
    for i in range(len(df_producto)):

        # Se comprueba si hoy se recibe algún pedido
        for pedido in pedidos_pendientes:
            if pedido[0] == i:
                stock_actual += pedido[1]
        
        # Se eliminan los pedidos recibidos
        pedidos_pendientes = [p for p in pedidos_pendientes if p[0] > i]
        
        # Se substrae la demanda real
        demanda_real: float = df_producto["udsVenta"].iloc[i]
        demanda_total += demanda_real
        
        ventas_reales: float = min(stock_actual, demanda_real)
        ventas_totales += ventas_reales
    
        ventas_perdidas: float = demanda_real - ventas_reales
        
        stock_actual -= ventas_reales
    
        # Cálculo del coste asociado a una rotura de stock
        coste_rotura_total += precio * ventas_perdidas
        
        # Se realiza un pedido si el día coincide con el intervalo de diasEntrePedidos
        if i % dias_entre_pedidos == 0:
            stock_pendiente: float = sum([p[1] for p in pedidos_pendientes])
            posicion_inventario: float = stock_actual + stock_pendiente
            
            stock_seguridad: float = (factor_servicio * sigma * np.sqrt(ciclo_aprov))
            demanda_prevista: float = df_producto[[f"pred_{h}" for h in range(1, ciclo_aprov + 1)]].iloc[i].sum()
            
            nivel_objetivo: float = stock_seguridad + demanda_prevista
            cantidad_pedido: float = max(0, nivel_objetivo - posicion_inventario)
    
            if cantidad_pedido > 0:
                dia_llegada: int = i + leadtime  # Se registra el día de la futura entrega
                pedidos_pendientes.append((dia_llegada, cantidad_pedido))
    
        # Cálculo del coste de inventario del día
        coste_stock_dia: float = 0.05 * precio * stock_actual
        coste_stock_total += coste_stock_dia

        # Se almacena la cantidad de inventario disponible al final del día
        unidades.append(stock_actual)  

    return unidades, {
        "coste_stock_total": coste_stock_total,
        "coste_rotura_total": coste_rotura_total,
        "ventas_totales": ventas_totales,
        "demanda_total": demanda_total
    }

<a id='ej2.1'></a>
## 2.1. Mejor modelo global

A continuación, se lanza una simulación de inventario con una política de reposición de tipo _Order-Up-To_ con los resultados del mejor modelo global.

In [ ]:
# Se lanza una simulación de tipo order-up-to (R, S) con los resultados del mejor modelo global de ML
nivel_servicio: float = 0.95
unidades_productos_ml_global: Dict[int, List[float]] = {}
metricas_productos_ml_global: Dict[int, Dict[str, float]] = {}

for producto, df_producto in mejor_modelo_global.groupby("producto"):
    rmses_producto: pd.DataFrame = rmses_mejor_modelo_global[rmses_mejor_modelo_global["producto"] == producto]
    unidades, metricas = simulacion_order_up_to(df_producto, rmses_producto, nivel_servicio)
    
    unidades_productos_ml_global[producto] = unidades
    metricas_productos_ml_global[producto] = metricas

In [ ]:
# Se agregan las cuatro métricas de evaluación de las simulaciones de las predicciones del mejor modelo global
coste_stock_total_ml_global: float = 0.0
coste_rotura_total_ml_global: float = 0.0
ventas_totales_ml_global: float = 0.0
demanda_total_ml_global: float = 0.0

for p, metricas in metricas_productos_ml_global.items():
    coste_stock_total_ml_global += metricas["coste_stock_total"]
    coste_rotura_total_ml_global += metricas["coste_rotura_total"]
    ventas_totales_ml_global += metricas["ventas_totales"]
    demanda_total_ml_global += metricas["demanda_total"]

In [ ]:
# A partir de las métricas anteriores, se calcula el coste total y la fracción de la demanda total satisfecha para todos
# los productos del dataset global
coste_total_ml_global: float = coste_stock_total_ml_global + coste_rotura_total_ml_global
pct_demanda_satisfecha_ml_global: float = ventas_totales_ml_global / demanda_total_ml_global if demanda_total_ml_global > 0 else 0

print("--- RESULTADOS DE LA EVALUACIÓN ECONÓMICA DEL MEJOR MODELO GLOBAL DE MACHINE LEARNING ---\n")
print(f"Coste asociado al inventario: {coste_stock_total_ml_global:.2f}")
print(f"Coste de las roturas de stock: {coste_rotura_total_ml_global:.2f}")
print(f"Coste total: {coste_total_ml_global:.2f}")
print(f"Porcentaje de la demanda satisfecha: {pct_demanda_satisfecha_ml_global:.4f}")

In [ ]:
# Se visualiza la demanda real frente junto con el nivel de inventario calculado en las simulaciones de tres productos aleatorios
N_PRODUCTOS: int = 3
seleccion_productos: List[int] = random.sample(mejor_modelo_global["producto"].unique().tolist(), N_PRODUCTOS)

fig, axs = plt.subplots(nrows=len(seleccion_productos), ncols=1, figsize=(10, 20))

for ax, producto in zip(axs, seleccion_productos):
    df_producto: pd.DataFrame = mejor_modelo_global[mejor_modelo_global["producto"] == producto]
    unidades_producto: List[float] = unidades_productos_ml_global[producto]
    
    ax.plot(df_producto["fecha"], df_producto["udsVenta"], label="demanda_real", alpha=0.5, color="blue")
    ax.plot(df_producto["fecha"], unidades_producto, label="inventario_ml", alpha=0.5, color="red")

    ax.set_title(f"Demanda real Vs Stock del producto {producto} del mejor modelo global")
    ax.set_xlabel("Días")
    ax.set_xticks([df_producto["fecha"].iloc[i] for i in range(0, len(df_producto["fecha"]), 20)])
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("Unidades")
    
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

<a id='ej2.2'></a>
## 2.2. Mejores modelos por _cluster_

Seguidamente, se emplea la función _**simulacion_order_up_to()**_ para evaluar el impacto económico siguiendo la estrategia de la segmentación de productos en _clusters_.

In [ ]:
# Se lanza una simulación de tipo order-up-to (R, S) con los resultados de los mejores modelos por cluster
nivel_servicio: float = 0.95
unidades_productos_ml_grupos: Dict[int, List[float]] = {}
metricas_productos_ml_grupos: Dict[int, Dict[str, float]] = {}

for producto, df_producto in mejores_modelos_grupos.groupby("producto"):
    rmses_producto: pd.DataFrame = rmses_mejores_modelos_grupos[rmses_mejores_modelos_grupos["producto"] == producto]
    unidades, metricas = simulacion_order_up_to(df_producto, rmses_producto, nivel_servicio)
    
    unidades_productos_ml_grupos[producto] = unidades
    metricas_productos_ml_grupos[producto] = metricas

In [ ]:
# Se agregan las cuatro métricas de evaluación de las simulaciones de las predicciones de los mejores modelos por cluster
coste_stock_total_ml_grupos: float = 0.0
coste_rotura_total_ml_grupos: float = 0.0
ventas_totales_ml_grupos: float = 0.0
demanda_total_ml_grupos: float = 0.0

for p, metricas in metricas_productos_ml_grupos.items():
    coste_stock_total_ml_grupos += metricas["coste_stock_total"]
    coste_rotura_total_ml_grupos += metricas["coste_rotura_total"]
    ventas_totales_ml_grupos += metricas["ventas_totales"]
    demanda_total_ml_grupos += metricas["demanda_total"]

In [ ]:
# A partir de las métricas anteriores, se calcula el coste total y la fracción de la demanda total satisfecha para todos
# los productos de los mejores modelos por cluster
coste_total_ml_grupos: float = coste_stock_total_ml_grupos + coste_rotura_total_ml_grupos
pct_demanda_satisfecha_ml_grupos: float = ventas_totales_ml_grupos / demanda_total_ml_grupos if demanda_total_ml_grupos > 0 else 0

print("--- RESULTADOS DE LA EVALUACIÓN ECONÓMICA DE LOS MEJORES MODELOS POR CLUSTER DE MACHINE LEARNING ---\n")
print(f"Coste asociado al inventario: {coste_stock_total_ml_grupos:.2f}")
print(f"Coste de las roturas de stock: {coste_rotura_total_ml_grupos:.2f}")
print(f"Coste total: {coste_total_ml_grupos:.2f}")
print(f"Porcentaje de la demanda satisfecha: {pct_demanda_satisfecha_ml_grupos:.4f}")

In [ ]:
# Se visualiza la demanda real frente junto con el nivel de inventario calculado en las simulaciones de tres productos aleatorios
N_PRODUCTOS: int = 3
seleccion_productos: List[int] = random.sample(mejores_modelos_grupos["producto"].unique().tolist(), N_PRODUCTOS)

fig, axs = plt.subplots(nrows=len(seleccion_productos), ncols=1, figsize=(10, 20))

for ax, producto in zip(axs, seleccion_productos):
    df_producto: pd.DataFrame = mejores_modelos_grupos[mejores_modelos_grupos["producto"] == producto]
    unidades_producto: List[float] = unidades_productos_ml_grupos[producto]
    
    ax.plot(df_producto["fecha"], df_producto["udsVenta"], label="demanda_real", alpha=0.5, color="blue")
    ax.plot(df_producto["fecha"], unidades_producto, label="inventario_ml", alpha=0.5, color="red")

    ax.set_title(f"Demanda real Vs Stock del producto {producto} de los mejores modelos por cluster")
    ax.set_xlabel("Días")
    ax.set_xticks([df_producto["fecha"].iloc[i] for i in range(0, len(df_producto["fecha"]), 20)])
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("Unidades")
    
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

<a id='ej2.3'></a>
## 2.3. Modelo _naive_

Finalmente, se lanza la simulación con los resultados del modelo _naive_.

In [ ]:
# Se lanza una simulación de tipo order-up-to (R, S) con los resultados del modelo naive
nivel_servicio: float = 0.95
modelo_naive: pd.DataFrame = modelos["naive_global"]
rmses_modelo_naive: pd.DataFrame = rmses_modelos["naive_global"]
unidades_productos_naive: Dict[int, List[float]] = {}
metricas_productos_naive: Dict[int, Dict[str, float]] = {}

for producto, df_producto in modelo_naive.groupby("producto"):
    rmses_producto: pd.DataFrame = rmses_modelo_naive[rmses_modelo_naive["producto"] == producto]
    unidades, metricas = simulacion_order_up_to(df_producto, rmses_producto, nivel_servicio)
    
    unidades_productos_naive[producto] = unidades
    metricas_productos_naive[producto] = metricas

In [ ]:
# Se agregan las cuatro métricas de evaluación de las simulaciones de las predicciones del modelo naive
coste_stock_total_naive: float = 0.0
coste_rotura_total_naive: float = 0.0
ventas_totales_naive: float = 0.0
demanda_total_naive: float = 0.0

for p, metricas in metricas_productos_naive.items():
    coste_stock_total_naive += metricas["coste_stock_total"]
    coste_rotura_total_naive += metricas["coste_rotura_total"]
    ventas_totales_naive += metricas["ventas_totales"]
    demanda_total_naive += metricas["demanda_total"]

In [ ]:
# A partir de las métricas anteriores, se calcula el coste total y la fracción de la demanda total satisfecha para todos
# los productos del dataset
coste_total_naive: float = coste_stock_total_naive + coste_rotura_total_naive
pct_demanda_satisfecha_naive: float = ventas_totales_naive / demanda_total_naive if demanda_total_naive > 0 else 0

print("--- RESULTADOS DE LA EVALUACIÓN ECONÓMICA DEL MODELO NAIVE ---\n")
print(f"Coste asociado al inventario: {coste_stock_total_naive:.2f}")
print(f"Coste de las roturas de stock: {coste_rotura_total_naive:.2f}")
print(f"Coste total: {coste_total_naive:.2f}")
print(f"Porcentaje de la demanda satisfecha: {pct_demanda_satisfecha_naive:.4f}")

In [ ]:
# Se visualiza la demanda real frente junto con el nivel de inventario calculado en las simulaciones de tres productos aleatorios
N_PRODUCTOS: int = 3
seleccion_productos: List[int] = random.sample(modelo_naive["producto"].unique().tolist(), N_PRODUCTOS)

fig, axs = plt.subplots(nrows=len(seleccion_productos), ncols=1, figsize=(10, 20))

for ax, producto in zip(axs, seleccion_productos):
    df_producto: pd.DataFrame = modelo_naive[modelo_naive["producto"] == producto]
    unidades_producto: List[float] = unidades_productos_naive[producto]
    
    ax.plot(df_producto["fecha"], df_producto["udsVenta"], label="demanda_real", alpha=0.5, color="blue")
    ax.plot(df_producto["fecha"], unidades_producto, label="inventario_naive", alpha=0.5, color="red")

    ax.set_title(f"Demanda real Vs Stock del producto {producto} del modelo naive")
    ax.set_xlabel("Días")
    ax.set_xticks([df_producto["fecha"].iloc[i] for i in range(0, len(df_producto["fecha"]), 20)])
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("Unidades")
    
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

<br><br><a id="ej3"></a>
# 3. Tabla resumen

En este apartado se proporciona una tabla resumen con las principales métricas calculadas de cada una de las tres simulaciones efectuadas para comparar el impacto económico que supone cada enfoque.

In [ ]:
# Se crea la tabla resumen de la evaluación económica
tabla_resumen: pd.DataFrame = pd.DataFrame({
    "Enfoque": ["Modelo global ML", "Modelos por cluster ML", "Modelo naive"],
    "Costes_stock": [coste_stock_total_ml_global, coste_stock_total_ml_grupos, coste_stock_total_naive],
    "Costes_stockouts": [coste_rotura_total_ml_global, coste_rotura_total_ml_grupos, coste_rotura_total_naive],
    "Costes_totales": [coste_total_ml_global, coste_total_ml_grupos, coste_total_naive],
    "Demanda_satisfecha": [pct_demanda_satisfecha_ml_global, pct_demanda_satisfecha_ml_grupos, pct_demanda_satisfecha_naive]
})

tabla_resumen["Costes_stock"] = tabla_resumen["Costes_stock"].astype(int)
tabla_resumen["Costes_stockouts"] = tabla_resumen["Costes_stockouts"].astype(int)
tabla_resumen["Costes_totales"] = tabla_resumen["Costes_totales"].astype(int)

tabla_resumen.round({"Demanda_satisfecha": 4}).head()

In [ ]:
# Distribución de los costes de almacenamiento
fig, ax = plt.subplots(figsize=(5, 6))

tabla_resumen.plot(
    kind="bar",
    x="Enfoque",
    y="Costes_stock",
    color="blue",
    alpha=0.6,
    edgecolor="black",
    linewidth=1,
    legend=False,
    ax=ax
)

ax.set_title("Distribución de costes de almacenamiento de stock")
ax.tick_params(axis="x", rotation=45)

ax.yaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f'{x/1e6:.0f}')
)
ax.set_ylabel("Coste (millones €)")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de los costes de roturas de inventario
fig, ax = plt.subplots(figsize=(5, 6))

tabla_resumen.plot(
    kind="bar",
    x="Enfoque",
    y="Costes_stockouts",
    color="blue",
    alpha=0.6,
    edgecolor="black",
    linewidth=1,
    legend=False,
    ax=ax
)

ax.set_title("Distribución de costes de los stockouts")
ax.tick_params(axis="x", rotation=45)

ax.yaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f'{x/1e5:.1f}')
)
ax.set_ylabel("Coste (cientos de miles €)")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de los costes totales
fig, ax = plt.subplots(figsize=(5, 6))

tabla_resumen.plot(
    kind="bar",
    x="Enfoque",
    y="Costes_totales",
    color="blue",
    alpha=0.6,
    edgecolor="black",
    linewidth=1,
    legend=False,
    ax=ax
)

ax.set_title("Distribución de los costes totales")
ax.tick_params(axis="x", rotation=45)

ax.yaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f'{x/1e6:.0f}')
)
ax.set_ylabel("Coste (millones €)")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Distribución del porcentaje de la demanda satisfecha
fig, ax = plt.subplots(figsize=(5, 6))

tabla_resumen.plot(
    kind="bar",
    x="Enfoque",
    y="Demanda_satisfecha",
    color="blue",
    alpha=0.6,
    edgecolor="black",
    linewidth=1,
    legend=False,
    ax=ax
)

ax.set_title("Porcentajes de la demanda satisfecha")
ax.tick_params(axis="x", rotation=45)
ax.set_ylabel("Demanda satisfecha (%)")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Se calcula el ahorro que supone la implementación de modelos predictivos de ML respecto a un modelo naive
ahorro_global_naive: int = (
    tabla_resumen.loc[tabla_resumen["Enfoque"] == "Modelo naive", "Costes_totales"].values[0]
    - tabla_resumen.loc[tabla_resumen["Enfoque"] == "Modelo global ML", "Costes_totales"].values[0]
)

ahorro_grupos_naive: int = (
    tabla_resumen.loc[tabla_resumen["Enfoque"] == "Modelo naive", "Costes_totales"].values[0]
    - tabla_resumen.loc[tabla_resumen["Enfoque"] == "Modelos por cluster ML", "Costes_totales"].values[0]
)

ahorro_global_grupos: int = (
    tabla_resumen.loc[tabla_resumen["Enfoque"] == "Modelos por cluster ML", "Costes_totales"].values[0]
    - tabla_resumen.loc[tabla_resumen["Enfoque"] == "Modelo global ML", "Costes_totales"].values[0]
)

print(
    "El uso de modelos globales de Machine Learning para la previsión de la demanda supone un ahorro "
    f"de {ahorro_global_naive}€ respecto al modelo naive"
)
print(
    "El uso de modelos de Machine Learning por clusters para la previsión de la demanda supone un ahorro "
    f"de {ahorro_grupos_naive}€ respecto al modelo naive"
)
print(
    f"El uso de un modelo global supone el ahorro de {ahorro_global_grupos}€ respecto a una estrategia de segmentación "
    "según las características de los productos"
)

<br><br>En términos generales, tanto el modelo global de _Machine Learning_ como el enfoque basado en modelos por _cluster_ muestran un desempeño claramente superior al modelo _naive_, lo que confirma la capacidad de los modelos predictivos avanzados para mejorar simultáneamente la eficiencia operativa y el nivel de servicio.

El modelo global de aprendizaje automático obtiene el menor coste total de almacenamiento y roturas de _stock_, alcanzando un valor de 6.035.671€, frente a los 6.319.052€ del enfoque por grupos, y los 11.255.072€ del modelo _naive_. Esto supone una reducción muy significativa respecto al enfoque base, cercana al 46 %, lo que evidencia el impacto económico que puede tener una previsión de demanda más precisa sobre la gestión del inventario. Esta mejora no solo se refleja en los costes agregados, sino también en el equilibrio alcanzado entre costes de almacenamiento y costes derivados de roturas de inventario. El modelo global presenta unos costes de almacenamiento inferiores a los del enfoque por _clusters_, aunque a cambio implica unos costes de _stockout_ ligeramente superiores. Esto indica que el modelo global adopta una política de inventario más ajustada y eficiente desde el punto de vista económico, manteniendo menores niveles medios de _stock_ sin comprometer de forma excesiva la disponibilidad de producto. Como consecuencia, el modelo global logra el mejor balance entre ambos tipos de costes.

Por otro lado, el enfoque basado en grupos consigue la mayor fracción de demanda satisfecha, hecho que denota una muy buena capacidad de respuesta ante la demanda real. Este comportamiento sugiere que la especialización de modelos según grupos de productos permite capturar patrones de demanda específicos, especialmente en artículos con dinámicas particulares o comportamientos heterogéneos. Sin embargo, esta mejora en el nivel de servicio se obtiene a costa de un incremento notable en los costes de almacenamiento. El enfoque por _clusters_ tiende a mantener inventarios más elevados para garantizar una mayor cobertura de la demanda, lo que incrementa el capital inmovilizado y los costes asociados al almacenamiento. En consecuencia, aunque consigue minimizar más eficazmente las roturas de _stock_, el impacto económico total termina siendo ligeramente peor que el del modelo global.
A pesar de ello, los resultados obtenidos sugieren que este enfoque tiene un gran potencial de futuro. En particular, uno de los aspectos más relevantes es el comportamiento observado en los productos clasificados dentro del grupo _top_ventas_, que contiene los ítems que concentran una parte muy significativa de la demanda total, donde pequeñas mejoras predictivas sobre ellos pueden traducirse en reducciones sustanciales de costes y aumentos importantes del nivel de servicio. Es razonable pensar que, si el rendimiento de los modelos especializados sobre este subconjunto de productos pudiera mejorarse, el enfoque por grupos podría superar al modelo global también en términos económicos. 

En este sentido, se podrían orientar futuras líneas de trabajo al desarrollo de modelos específicos para los productos de mayor demanda, incorporando variables adicionales capaces de describir mejor su comportamiento temporal y comercial. Por ejemplo, podrían añadirse variables promocionales más detalladas, atributos relacionados con tendencias de consumo, eventos externos o campañas comerciales, así como información histórica agregada mediante estadísticas móviles más sofisticadas. Otra posible línea de mejora consiste en redefinir el propio procedimiento de agrupamiento de productos. Por ejemplo, podrían emplearse técnicas de _clustering_ más avanzadas, partiendo de criterios de segmentación orientados directamente al impacto económico, de forma que podrían generarse grupos más homogéneos y favorecer un mejor aprendizaje de los patrones de demanda. Esto mismo podría aplicarse también al propio entrenamiento de los modelos, de forma que el aprendizaje se realice minimizando una métrica relacionada directamente con el impacto económico del algoritmo.

A modo de conclusión, los resultados muestran que el enfoque global constituye actualmente la alternativa más eficiente desde una perspectiva estrictamente económica, al ofrecer el menor coste total. No obstante, el enfoque basado en _clusters_ demuestra una capacidad muy prometedora para mejorar el nivel de servicio y reducir las roturas de _tock_, especialmente en segmentos de productos críticos. Consecuentemente, si se implementan mejoras centradas en los productos de mayor impacto comercial, se podría lograr que los modelos especializados superasen al enfoque global también en términos de rentabilidad económica.